# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

   order_id           item  category  qty   price
0         0    Rain Poncho  RainGear  3.0  $12.00
1         1    foam finger   Apparel  1.0     7.5
2         2  cheese burger     Merch  1.0   $7.50
3         3   Cheeseburger      Food  NaN   $7.50
4         4  cheese burger   Apparel  1.0     7.5

### TODO 1 — drop duplicates

In [3]:
# Remove exact repeated exports, then copy so later assignments are explicit.
duplicate_rows = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
log('duplicates', 'dropped exact duplicate rows', duplicate_rows)

[duplicates] dropped exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
# Remove currency formatting before converting every price to float.
df['price'] = (df['price'].astype(str)
               .str.replace('$', '', regex=False)
               .str.replace(',', '', regex=False)
               .str.strip()
               .astype(float))
log('price', 'removed currency formatting and converted price to float', len(df))

[price] removed currency formatting and converted price to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# Invalid and negative quantities cannot count as completed sales in this gross-sales table.
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
missing_qty = int(df['qty'].isna().sum())
negative_qty = int((df['qty'] < 0).sum())
df = df.loc[df['qty'].notna() & (df['qty'] > 0)].copy()
df['qty'] = df['qty'].astype(int)
log('qty', 'dropped rows with missing quantities', missing_qty)
log('qty', 'dropped negative quantities from gross sales', negative_qty)

[qty] dropped rows with missing quantities (12 row(s))
[qty] dropped negative quantities from gross sales (13 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print('item variants before cleaning:')
print(df['item'].value_counts())

# The six spellings represent three products.
ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho',
}
before_items = df['item'].nunique()
df['item'] = df['item'].replace(ITEM_MAP)
log('item', f'collapsed {before_items} spellings into {df["item"].nunique()} products', len(df))

item variants before cleaning:
item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] collapsed 6 spellings into 3 products (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print('category variants before cleaning:')
print(df['category'].value_counts())

df['category'] = (df['category'].str.strip().str.lower()
                  .str.replace('-', '', regex=False))
CATEGORY_MAP = {
    'food': 'Food',
    'merch': 'Merch',
    'apparel': 'Merch',  # Business decision: Apparel rolls up into Merch.
    'raingear': 'RainGear',
}
before_categories = df['category'].nunique()
df['category'] = df['category'].replace(CATEGORY_MAP)
log('category',
    f'normalized labels and combined Apparel with Merch; {before_categories} became {df["category"].nunique()}',
    len(df))

category variants before cleaning:
category
Merch        51
Food         51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] normalized labels and combined Apparel with Merch; 4 became 3 (275 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
# These checks make the cleaning rules executable and catch regressions.
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item'].unique()) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
assert set(df['category'].unique()) == {'Food', 'Merch', 'RainGear'}
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
# Calculate row revenue, then summarize the business result by category.
df['revenue'] = df['qty'] * df['price']
revenue_by_category = (df.groupby('category')['revenue']
                       .sum()
                       .sort_values(ascending=False)
                       .round(2))

print(revenue_by_category)
print(f'overall total: ${df["revenue"].sum():,.2f}')
print(f'Vendors should stock more {revenue_by_category.index[0]} because it generated the most revenue.')

category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
overall total: $4,740.00
Vendors should stock more Food because it generated the most revenue.


**What I would tell the vendor:** _..._

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

         step                                           decision  rows
0  duplicates                       dropped exact duplicate rows    15
1       price  removed currency formatting and converted pric...   300
2         qty               dropped rows with missing quantities    12
3         qty       dropped negative quantities from gross sales    13
4        item              collapsed 6 spellings into 3 products   275
5    category  normalized labels and combined Apparel with Me...   275

### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

_your answer here_